# Step 6 — SMOTE + Permutation Feature Importance

This notebook uses SMOTE to balance classes, then computes Permutation Feature Importance (PFI)
to identify the most predictive genes. PFI measures the drop in model performance when each
feature is randomly shuffled.

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from sklearn.model_selection import train_test_split

from asd_pipeline_utils import (
    RANDOM_STATE, SMOTE_PFI_PATH,
    get_targets, load_analysis_frame, load_signature_stability,
    split_features_and_metadata, build_smote_pipeline, compute_pfi,
)

In [ ]:
# Load data and signature stability from Step 5
final_df = load_analysis_frame()
X, meta = split_features_and_metadata(final_df)
_, y_multi = get_targets(meta)
stability_df = load_signature_stability()

# Select top 100 stable genes
step6_genes = stability_df.head(100)["gene"].tolist()
X_step6 = X[step6_genes].copy()

print(f"Step 6: {len(step6_genes)} stable genes, matrix {X_step6.shape}")
print(f"Class distribution:\n{y_multi.value_counts()}\n")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_step6, y_multi, test_size=0.2, stratify=y_multi, random_state=RANDOM_STATE
)

# Fit SMOTE + LR pipeline
pipe = build_smote_pipeline()
pipe.fit(X_train, y_train)
print(f"SMOTE + LR — Train acc: {pipe.score(X_train, y_train):.3f}, Test acc: {pipe.score(X_test, y_test):.3f}")

# Permutation Feature Importance
pfi_df = compute_pfi(pipe, X_test, y_test)

print(f"\n=== Permutation Feature Importance (top 25) ===")
print(pfi_df.head(25).to_string(index=False))

significant = pfi_df[pfi_df["importance_mean"] > 0]
print(f"\nGenes with positive PFI: {len(significant)}")

# Save
pfi_df.to_csv(SMOTE_PFI_PATH, index=False)
print(f"\nSaved to {SMOTE_PFI_PATH}")
pfi_df.head(25)